# 🚀 OmniGen AI Studio - Google Colab 16GB Cloud GPU Server
**Free Tesla T4 / A100 GPU • 0% Laptop Resource Usage • Multi-Model & LoRA Engine**

### Instructions:
1. Go to **Runtime -> Change runtime type -> Select T4 GPU**.
2. Run all cells below.
3. Copy the public **Cloudflare Tunnel URL** (`https://xxxx.trycloudflare.com`).
4. In OmniGen Web App, click the **Settings / Backend** icon in the header, paste the URL, and click **Test Ping**!

In [1]:
# Cell 1: Install Dependencies
!pip install -q diffusers transformers accelerate safetensors sentencepiece protobuf fastapi uvicorn pydantic pycloudflared nest_asyncio python-multipart peft

In [2]:
# Cell 2: Download Models & SDXL Lightning Accelerator
import os
os.makedirs("/content/Models", exist_ok=True)
os.makedirs("/content/LoRAs", exist_ok=True)

# Define the path for the desired Civitai base model
BASE_MODEL_PATH = "/content/Models/crucibleRINGPonyxl_v28.safetensors"
LIGHTNING_PATH = "/content/LoRAs/sdxl_lightning_4step_lora.safetensors"

# Import userdata for secrets
from google.colab import userdata

try:
    CIVITAI_API_KEY = userdata.get('CIVITAI_API_KEY')
except Exception:
    CIVITAI_API_KEY = None

def civitai_download_url(model_version_id):
    base_url = f"https://civitai.com/api/download/models/{model_version_id}"
    if CIVITAI_API_KEY:
        return f"{base_url}?token={CIVITAI_API_KEY}"
    return base_url

# Clean up corrupted files
if os.path.exists(BASE_MODEL_PATH) and os.path.getsize(BASE_MODEL_PATH) < 1024 * 1024:
    print(f"⚠️ Found corrupted or incomplete file at {BASE_MODEL_PATH}. Deleting and re-downloading...")
    os.remove(BASE_MODEL_PATH)

if not os.path.exists(BASE_MODEL_PATH):
    print("📥 Downloading CrucibleRING PonyXL v28 (~6.6GB) from Civitai...")
    !wget -c "{civitai_download_url('1979291')}" -O {BASE_MODEL_PATH}
    if os.path.exists(BASE_MODEL_PATH) and os.path.getsize(BASE_MODEL_PATH) < 1024 * 1024:
        print(f"⚠️ Download failed. It might be corrupted. Deleting file.")
        os.remove(BASE_MODEL_PATH)
    else:
        print(f"✅ Base model downloaded successfully.")

if not os.path.exists(LIGHTNING_PATH):
    print("⚡ Downloading SDXL Lightning 4-Step LoRA...")
    !wget -c "https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_4step_lora.safetensors" -O {LIGHTNING_PATH}

print("✅ Storage ready! Models directory is prepared.")


📥 Downloading CrucibleRING PonyXL v28 (~6.6GB)...
--2026-08-27 21:07:19--  https://huggingface.co/Lies/crucibleRINGPonyxl_v28/resolve/main/crucibleRINGPonyxl_v28.safetensors
Resolving huggingface.co (huggingface.co)... 99.86.101.56, 99.86.101.64, 99.86.101.39, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.56|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized

Username/Password Authentication Failed.
⚡ Downloading SDXL Lightning 4-Step Accelerator...
--2026-08-27 21:07:19--  https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_4step_lora.safetensors
Resolving huggingface.co (huggingface.co)... 99.86.101.39, 99.86.101.56, 99.86.101.64, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.39|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65d505951e2597ff935d1be2/59bb0d054a29afeb6c71c26e0c4fdcff244c9abe22cfea8139f67ed912383d81?user_id=public&X-Xet-Cas-

In [3]:
# Cell 3: Load Model into 16GB Cloud VRAM
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

print("🧹 Cleaning up /content/Models from non-base model safetensors files...")
for filename in os.listdir("/content/Models"):
    file_path = os.path.join("/content/Models", filename)
    if filename.endswith(".safetensors") and file_path != BASE_MODEL_PATH:
        print(f"    Deleting file: {file_path}")
        os.remove(file_path)

print(f"🚀 Initializing Pipeline on {torch.cuda.get_device_name(0)}...")
CURRENT_BASE_MODEL_FILE = os.path.basename(BASE_MODEL_PATH)
global pipe

pipe = StableDiffusionXLPipeline.from_single_file(
    BASE_MODEL_PATH,
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
print("✅ Base model loaded successfully and ready for dynamic UI injection.")


🚀 Loading SDXL Pipeline on Tesla T4 (16GB VRAM)...


OSError: Unable to load weights from checkpoint file for '/content/Models/crucibleRINGPonyxl_v28.safetensors' at '/content/Models/crucibleRINGPonyxl_v28.safetensors'. 

In [ ]:
# Cell 4: Launch FastAPI Server & Cloudflare Public Tunnel
import io, base64, time, json, threading, nest_asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional, Union
import uvicorn
from pycloudflared import try_cloudflare
import requests
import gc

nest_asyncio.apply()
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class Txt2ImgRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "score_4, score_5, score_6, bad hands, blurry, low quality"
    steps: Optional[int] = 20
    cfg_scale: Optional[float] = 6.5
    width: Optional[int] = 832
    height: Optional[int] = 1216
    seed: Optional[int] = -1
    base_model: Optional[Union[dict, str]] = "crucibleRINGPonyxl_v28.safetensors"
    loras: Optional[List[Union[dict, str]]] = []
    civitai_api_key: Optional[str] = ""

def download_civitai_model(download_url, dest_path, api_key):
    if os.path.exists(dest_path): return True
    print(f"📥 Downloading missing model to {os.path.basename(dest_path)}...")
    active_key = api_key if api_key else CIVITAI_API_KEY
    headers = {}
    if active_key: headers["Authorization"] = f"Bearer {active_key}"
    try:
        response = requests.get(download_url, headers=headers, stream=True)
        if response.status_code == 200:
            with open(dest_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            if os.path.getsize(dest_path) < 1024 * 1024:
                os.remove(dest_path)
                return False
            return True
        return False
    except Exception as e:
        return False

@app.get("/")
def health():
    return {
        "status": "online",
        "gpu": torch.cuda.get_device_name(0),
        "base_model": CURRENT_BASE_MODEL_FILE
    }

@app.post("/sdapi/v1/txt2img")
def txt2img(req: Txt2ImgRequest):
    global CURRENT_BASE_MODEL_FILE, pipe
    seed = req.seed if (req.seed is not None and req.seed >= 0) else int(torch.randint(0, 2**32, (1,)).item())
    generator = torch.Generator("cuda").manual_seed(seed)

    req_base_model_file = CURRENT_BASE_MODEL_FILE
    req_base_model_url = None
    req_architecture = "SDXL 1.0"
    if isinstance(req.base_model, dict):
        req_architecture = req.base_model.get("architecture", "SDXL 1.0")
        req_base_model_file = req.base_model.get("fileName", req_base_model_file)
        req_base_model_url = req.base_model.get("downloadUrl")
    elif isinstance(req.base_model, str) and req.base_model:
        req_base_model_file = req.base_model

    if req_base_model_file != CURRENT_BASE_MODEL_FILE:
        model_path = os.path.join("/content/Models", req_base_model_file)
        if not os.path.exists(model_path):
            if req_base_model_url:
                success = download_civitai_model(req_base_model_url, model_path, req.civitai_api_key)
                if not success: return {"error": f"Failed to download {req_base_model_file}. Check CivitAI API Key."}
            else:
                return {"error": f"Model {req_base_model_file} not found locally and no download URL."}

        print(f"🔄 Swapping base model from {CURRENT_BASE_MODEL_FILE} to {req_base_model_file} (Arch: {req_architecture})...")
        if 'pipe' in globals() and pipe is not None:
            del pipe
        gc.collect()
        torch.cuda.empty_cache()

        if "SD 1.5" in req_architecture or "SD 1.4" in req_architecture:
            from diffusers import StableDiffusionPipeline
            pipe = StableDiffusionPipeline.from_single_file(model_path, torch_dtype=torch.float16, use_safetensors=True).to("cuda")
            pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
        elif "Flux" in req_architecture:
            from diffusers import FluxPipeline
            pipe = FluxPipeline.from_single_file(model_path, torch_dtype=torch.bfloat16).to("cuda")
            pipe.enable_model_cpu_offload()
        else:
            from diffusers import StableDiffusionXLPipeline
            pipe = StableDiffusionXLPipeline.from_single_file(model_path, torch_dtype=torch.float16, use_safetensors=True).to("cuda")
            pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

        CURRENT_BASE_MODEL_FILE = req_base_model_file

    loaded_adapters = []
    loaded_weights = []
    if req.loras:
        for item in req.loras:
            name = item if isinstance(item, str) else item.get("fileName") or item.get("name")
            weight = 0.85 if isinstance(item, str) else float(item.get("weight", 0.85))
            if not name: continue

            lora_file = name if name.endswith(".safetensors") else f"{name}.safetensors"
            lora_path = os.path.join("/content/LoRAs", lora_file)
            lora_url = item.get("downloadUrl") if isinstance(item, dict) else None

            if not os.path.exists(lora_path) and lora_url:
                download_civitai_model(lora_url, lora_path, req.civitai_api_key)

            if os.path.exists(lora_path) and lora_file != CURRENT_BASE_MODEL_FILE and lora_file != os.path.basename(LIGHTNING_PATH):
                try:
                    adapter_id = f"lora_{len(loaded_adapters)}"
                    pipe.load_lora_weights("/content/LoRAs", weight_name=lora_file, adapter_name=adapter_id)
                    loaded_weights.append(weight)
                    loaded_adapters.append(adapter_id)
                except Exception as e:
                    print(f"LoRA load note: {e}")

        if loaded_adapters:
            print(f"Activating adapters: {loaded_adapters} with weights {loaded_weights}")
            pipe.set_adapters(loaded_adapters, adapter_weights=loaded_weights)

    prompt_str = req.prompt if "score_" in req.prompt else f"score_9, score_8_up, score_7_up, source_anime, {req.prompt}"

    with torch.inference_mode():
        if "Flux" in str(type(pipe)):
            image = pipe(
                prompt=prompt_str,
                num_inference_steps=req.steps,
                guidance_scale=req.cfg_scale,
                width=req.width,
                height=req.height,
                generator=generator
            ).images[0]
        else:
            image = pipe(
                prompt=prompt_str,
                negative_prompt=req.negative_prompt,
                num_inference_steps=req.steps,
                guidance_scale=req.cfg_scale,
                width=req.width,
                height=req.height,
                generator=generator
            ).images[0]

    if loaded_adapters:
        try:
            pipe.delete_adapters(loaded_adapters)
        except Exception:
            pass

    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return {
        "images": [base64.b64encode(buf.getvalue()).decode("utf-8")],
        "source": f"Google Colab Cloud GPU ({torch.cuda.get_device_name(0)})"
    }

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)
tunnel = try_cloudflare(port=8000)
print(f"\n🎉 COPY THIS URL: {tunnel.tunnel}\n")
